# Level 5 - The artificial kidney: membranes + ion exchange
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/kidney-sim/blob/main/notebooks/05_artificial_kidney.ipynb)

**Goal:** model a dialysis machine. Here the idea of *ion exchange* is exactly how the machine works.

In [ ]:
# --- Setup: run this cell first --------------------------------------------
import os, sys, subprocess
REPO_URL = "https://github.com/YOUR-USERNAME/kidney-sim.git"   # change if you forked the project
if "google.colab" in sys.modules and not os.path.exists("kidneysim"):
    if not os.path.exists("kidney-sim"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("kidney-sim")
for path in [".", ".."]:
    if os.path.exists(os.path.join(path, "kidneysim")):
        sys.path.insert(0, os.path.abspath(path))
        break

import numpy as np
import matplotlib.pyplot as plt
import kidneysim as ks
from kidneysim import viz2d, viz3d
from kidneysim.ui import sliders
print("kidneysim", ks.__version__, "| interactive 3D (plotly):", viz3d.HAS_PLOTLY)

## 1. The dialyzer: diffusion through a membrane

Blood and dialysate flow in **opposite** directions (countercurrent, like the loop of Henle!):

$$Q_b\frac{dC_b}{dx} = -K_oA\,(C_b - C_d) \qquad Q_d\frac{dC_d}{dx} = -K_oA\,(C_b - C_d)$$

Solution (Michaels): $\;Z = Q_b/Q_d,\; NTU = K_oA/Q_b$

$$E = \frac{1-e^{NTU(1-Z)}}{Z-e^{NTU(1-Z)}}, \qquad K_D = E\,Q_b$$

Water is removed by **pressure** (ultrafiltration): $Q_{uf} = K_{uf}\cdot TMP$.

In [ ]:
prof = ks.dialysis.dialyzer_profile(Qb=0.3, Qd=0.5, KoA=1.0, C_in=25)
print(f"clearance = {ks.dialysis.dialyzer_clearance()*1000:.0f} mL/min "
      f"(healthy kidneys: ~56 mL/min for urea, but they work 24 h/day)")
print(f"ultrafiltration at TMP 80 mmHg: {ks.dialysis.ultrafiltration(80):.2f} L/h")
viz2d.plot_dialyzer(prof);

In [ ]:
viz3d.show(viz3d.dialyzer_3d(prof))

## 2. The patient: 3 sessions of 4 hours per week

Level 0 again, with $K = K_D + K_{residual}$ during dialysis. Dose: $Kt/V$ (target ≥ 1.2) and
urea reduction ratio $URR = 1 - C_{end}/C_{start}$ (target ≥ 65 %).

In [ ]:
tr = ks.dialysis.treatment(weeks=2)
viz2d.plot_treatment(tr);

## 3. Sorbent cartridge: real ION EXCHANGE

Urease converts urea to ammonium, then zirconium phosphate **exchanges** it for sodium:

$$\text{urea} + H_2O \xrightarrow{urease} 2\,NH_4^+ + CO_3^{2-} \qquad \text{ZrP–Na} + NH_4^+ \rightleftharpoons \text{ZrP–NH}_4 + Na^+$$

Column model (advection + exchange):

$$\varepsilon\frac{\partial c}{\partial t} + u\frac{\partial c}{\partial z} = -\rho_b\frac{\partial q}{\partial t}, \qquad
\frac{\partial q}{\partial t} = k\,(q_{eq}(c) - q), \qquad q_{eq} = \frac{q_{max}K c}{1+Kc}$$

In [ ]:
col = ks.dialysis.ion_exchange_column(c_in=15, Q=0.25, t_end_min=480)
t_bt = ks.dialysis.breakthrough_time(col)
print(f"resin capacity {col['capacity_mmol']:.0f} mmol NH4+, breakthrough after {t_bt:.0f} min")
viz2d.plot_breakthrough(col, t_bt);

In [ ]:
viz3d.show(viz3d.column_surface(col, which="c"))

In [ ]:
def experiment(blood_flow=300, dialysate_flow=500, KoA=1000, session_hours=4.0):
    tr = ks.dialysis.treatment(weeks=2, Qb=blood_flow/1000, Qd=dialysate_flow/1000,
                               KoA=KoA/1000, session_h=session_hours)
    print(f"dialyzer clearance {tr['K_D']*1000:.0f} mL/min, Kt/V = {tr['Kt_V'][0]:.2f}")
    viz2d.plot_treatment(tr)
    plt.show()

sliders(experiment, blood_flow=(100, 500, 25, 300), dialysate_flow=(300, 800, 50, 500),
        KoA=(300, 1500, 50, 1000), session_hours=(2, 8, 0.5, 4))

## Questions
1. Why is the countercurrent arrangement better than flowing blood and dialysate in the same direction?
2. Doubling the blood flow does **not** double the clearance. Why?
3. The sorbent releases Na⁺ for every NH₄⁺ captured. Why could that be a problem for the patient?
4. Compare: what does a real kidney do that this machine cannot? (Think of Levels 2–4.)